<div class="blog-language-switch" role="group" aria-label="文章语言">
<a href="../../Deep-Learning/12-pretraining-transfer-parameter-efficient-adaptation.html" lang="en" hreflang="en">English</a>
<span aria-current="page">中文</span>
</div>

[返回深度学习总览](Deep-Learning.html)

## **预训练、迁移与参数高效适配** {#pretraining-transfer-parameter-efficient-adaptation}

如果每个模型都从随机初始化开始训练，就等于假定各项任务彼此无关。**预训练（pretraining）**则先在源目标上学习参数，再把它们作为信息更充分的起点复用。**迁移学习（transfer learning）**询问哪些知识对目标任务或目标域仍然有用。**适配（adaptation）**则在少量标签、有限内存、多个客户或必须保留旧行为等约束下，决定应该更新哪些部分。因此，核心问题不只是“是否应该 fine-tune 这个模型”，而是：**任务特定的变化应该发生在哪里、多少变化才有证据支撑，以及如何发现有害变化？**

本章承接第 11 章的表示学习，但改变实验问题。我们先在干净手写数字上预训练分类器；部署时输入分布发生变化，而类别语义保持不变：数字向右移动一个像素、轻微模糊、对比度下降，并带有传感器噪声。目标域每类只有 8 个标注样本。所有方法使用相同的预训练 checkpoint、目标域划分、验证规则和评估指标。

数据来自 scikit-learn 收录的 [UCI Optical Recognition of Handwritten Digits dataset](https://archive.ics.uci.edu/dataset/80/optical%2Brecognition%2Bof%2B)：共 1,797 张 8×8 图像，DOI 为 [10.24432/C50P49](https://doi.org/10.24432/C50P49)，采用 CC BY 4.0 许可。这个紧凑实验用于研究**机制**，不是 state-of-the-art benchmark；它让我们能够在 CPU 上检查参数更新和数据分布边界。

![干净源域数字及其保持标签不变的目标域变换版本。](assets/dl12-source-target-shift.svg){fig-align="center" width="76%" fig-alt="从类别零到九的干净手写数字，以及经过平移、模糊和加噪后的配对图像。"}

*根据本章使用的 UCI 衍生数字样本生成的原创可视化。右侧应用了与代码相同的、保持标签不变的确定性变换族。*

<details>
<summary><strong>PyTorch：建立源域 checkpoint 与低样本目标域</strong></summary>

```python
import copy
import math
import random

import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.set_num_threads(1)


def seed_everything(seed=1212):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_everything()
digits = load_digits()
all_images = torch.tensor(digits.images, dtype=torch.float32).unsqueeze(1) / 16.0
all_labels = torch.tensor(digits.target, dtype=torch.long)
all_indices = np.arange(len(all_images))

source_train_idx, holdout_idx = train_test_split(
    all_indices, test_size=0.30, random_state=1212, stratify=digits.target
)
val_idx, test_idx = train_test_split(
    holdout_idx, test_size=0.50, random_state=1212,
    stratify=digits.target[holdout_idx],
)
source_train_x = all_images[source_train_idx]
source_train_y = all_labels[source_train_idx]
source_val_x, source_val_y = all_images[val_idx], all_labels[val_idx]
source_test_x, source_test_y = all_images[test_idx], all_labels[test_idx]


def make_target_domain(images, seed):
    # The mapping changes acquisition style, not the intended digit label.
    shifted = torch.zeros_like(images)
    shifted[:, :, :, 1:] = images[:, :, :, :-1]
    blurred = F.avg_pool2d(shifted, kernel_size=3, stride=1, padding=1)
    generator = torch.Generator().manual_seed(seed)
    noise = 0.055 * torch.randn(images.shape, generator=generator)
    return (0.82 * blurred + noise).clamp(0.0, 1.0)


target_pool_x = make_target_domain(source_train_x, seed=1213)
target_val_x = make_target_domain(source_val_x, seed=1214)
target_test_x = make_target_domain(source_test_x, seed=1215)


def balanced_low_shot_indices(labels, per_class=8, seed=1212):
    generator = torch.Generator().manual_seed(seed)
    selected = []
    for label in range(10):
        candidates = torch.where(labels == label)[0]
        selected.append(candidates[torch.randperm(len(candidates), generator=generator)[:per_class]])
    return torch.cat(selected)


target_adapt_local_idx = balanced_low_shot_indices(source_train_y, per_class=8)
target_adapt_x = target_pool_x[target_adapt_local_idx]
target_adapt_y = source_train_y[target_adapt_local_idx]


class DigitBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(64, 128)
        self.fc2 = nn.Linear(128, 64)

    def forward(self, images):
        flat = images.flatten(1)
        return F.relu(self.fc2(F.relu(self.fc1(flat))))


class DigitClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = DigitBackbone()
        self.head = nn.Linear(64, 10)

    def forward(self, images, return_features=False):
        features = self.backbone(images)
        logits = self.head(features)
        return (logits, features) if return_features else logits


def make_loader(images, labels, batch_size=128, shuffle=True, seed=1212):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        TensorDataset(images, labels), batch_size=batch_size, shuffle=shuffle,
        generator=generator,
    )


@torch.no_grad()
def accuracy(model, images, labels):
    model.eval()
    return float((model(images).argmax(dim=1) == labels).float().mean())


def train_with_validation(model, train_x, train_y, val_x, val_y, *, epochs=60,
                          lr=3e-3, weight_decay=1e-4, parameter_groups=None,
                          seed=1212):
    seed_everything(seed)
    trainable = [p for p in model.parameters() if p.requires_grad]
    groups = parameter_groups if parameter_groups is not None else trainable
    optimizer = torch.optim.AdamW(groups, lr=lr, weight_decay=weight_decay)
    loader = make_loader(train_x, train_y, batch_size=min(128, len(train_x)), seed=seed)
    best_state, best_val = copy.deepcopy(model.state_dict()), -1.0
    for _ in range(epochs):
        model.train()
        for batch_x, batch_y in loader:
            optimizer.zero_grad()
            loss = F.cross_entropy(model(batch_x), batch_y)
            loss.backward()
            optimizer.step()
        score = accuracy(model, val_x, val_y)
        if score > best_val:
            best_val, best_state = score, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return best_val


source_model = DigitClassifier()
train_with_validation(
    source_model, source_train_x, source_train_y, source_val_x, source_val_y,
    epochs=55, lr=3e-3, weight_decay=1e-4, seed=1212,
)
source_state = copy.deepcopy(source_model.state_dict())
source_clean_accuracy = accuracy(source_model, source_test_x, source_test_y)
zero_shot_target_accuracy = accuracy(source_model, target_test_x, source_test_y)
adaptation_results = {}

assert all_images.shape == (1797, 1, 8, 8)
assert len(set(source_train_idx) & set(test_idx)) == 0
assert torch.bincount(target_adapt_y).tolist() == [8] * 10
assert source_clean_accuracy > 0.90
assert zero_shot_target_accuracy < source_clean_accuracy
print({
    "split": (len(source_train_idx), len(val_idx), len(test_idx)),
    "target labels used": len(target_adapt_y),
    "source clean accuracy": round(source_clean_accuracy, 3),
    "zero-shot target accuracy": round(zero_shot_target_accuracy, 3),
})
```

</details>

目标验证集和测试集分别由原始验证索引与测试索引变换得到，从不使用训练样本。模型选择使用目标验证标签；目标测试集只用于最终报告。目标适配集来自源训练索引，因此任何 held-out 图像的变换副本都不会泄漏进训练过程。

### **为什么预训练能够迁移** {#why-pretraining-transfers}

当源参数编码了可以复用的计算时，预训练就能产生迁移价值。较早的视觉层可能检测局部强度变化，较深的层则把它们组合成笔画和数字部件。如果目标任务仍依赖这些因素，从源 checkpoint 开始优化就相当于从参数空间中的有用区域出发。与随机初始化相比，迁移可以减少标签需求、缩短优化过程，并提高稳定性。

这种收益是有条件的。设预训练模型可以分解为表示 $h=f_{\theta}(x)$ 和任务 head $\hat y=g_{\phi}(h)$，则目标风险为

$$
\mathcal{R}_T(\theta,\phi)=\mathbb{E}_{(x,y)\sim P_T}\left[\ell(g_{\phi}(f_{\theta}(x)),y)\right].
$$

只有当源 checkpoint 的表示让目标风险更容易降低时，它才真正有帮助。输入外观相似既不是必要条件，也不是充分条件：两个域的外观可能不同却共享因果特征，也可能看起来相似但标签含义已经变化。当源域 invariance 抹掉了目标信息、源域 shortcut 只在旧分布上有效，或激进更新在少量目标样本上过拟合时，就会发生负迁移（negative transfer）。

[Yosinski 等人](https://arxiv.org/abs/1411.1792)区分了特征通用性与 co-adaptation，并通过实验说明：随着任务差异扩大和高层专门化增强，特征可迁移性会下降。实际诊断因此应该比较 frozen probe、部分解冻和完整 fine-tuning。强大的 frozen probe 表明目标信息已经可以被线性访问；完整 fine-tuning 带来明显提升，说明表示存在不匹配；两者都很差，则可能意味着源任务、架构或数据并不合适。

迁移主要受三个变量支配：

- **任务相关性：**目标决策是否依赖预训练表示所保留的因素。
- **域偏移：**$P_S(x)$ 与 $P_T(x)$ 的差异是否属于该表示能够处理的类型。
- **目标证据：**必须有足够标签或自监督信号，才能支撑大量参数发生变化。

因此，上面的源域/目标域准确率差距本身就是证据。干净 checkpoint 已经掌握数字语义，但它的 pixel-to-feature 映射对部署时的 corruption 并不具有 invariance。接下来的方法会逐渐增加适配容量，以尝试缩小这一差距。

### **特征提取与完整微调** {#feature-extraction-full-fine-tuning}

**特征提取（feature extraction）**冻结 $\theta$，只更新目标 head $\phi$。它成本低、稳定，并允许多个任务特定 head 共享同一个 backbone；但其上限受制于 $f_{\theta}(x)$ 中已经可以访问的信息。**完整微调（full fine-tuning）**同时更新 $\theta$ 与 $\phi$，允许表示本身向目标域移动，却也增加 optimizer memory、过拟合风险和每任务 checkpoint 存储成本。

![从冻结 linear probe 到完整 fine-tuning 的迁移谱。](assets/dl12-transfer-spectrum.svg){fig-align="center" width="76%" fig-alt="一条水平迁移谱从稳定到可塑依次展示 linear probing、部分解冻、PEFT 与完整 fine-tuning。"}

*综合本章所比较迁移策略绘制的原创教学图。*

公平比较必须固定初始化和目标证据。下面两个模型都从 `source_state` 开始，也都没有获得额外目标样本。因为数字标签空间没有变化，所以保留预训练 head 并允许其适配；重新初始化 head 回答的是“任务语义已经改变”这一不同问题。

<details>
<summary><strong>PyTorch：比较冻结 backbone 与完整 fine-tuning</strong></summary>

```python
def count_parameters(model, trainable_only=False):
    parameters = (p for p in model.parameters() if (p.requires_grad or not trainable_only))
    return sum(p.numel() for p in parameters)


def record_method(name, model, task_parameters=None, note=""):
    adaptation_results[name] = {
        "target_accuracy": accuracy(model, target_test_x, source_test_y),
        "source_accuracy_active": accuracy(model, source_test_x, source_test_y),
        "trainable_parameters": count_parameters(model, trainable_only=True),
        "task_parameters": task_parameters if task_parameters is not None else count_parameters(model, True),
        "note": note,
    }


feature_extractor = DigitClassifier()
feature_extractor.load_state_dict(source_state)
for parameter in feature_extractor.backbone.parameters():
    parameter.requires_grad = False
train_with_validation(
    feature_extractor, target_adapt_x, target_adapt_y, target_val_x, source_val_y,
    epochs=70, lr=8e-3, weight_decay=1e-4, seed=1220,
)
record_method("Frozen features", feature_extractor, note="head only")

full_finetune = DigitClassifier()
full_finetune.load_state_dict(source_state)
train_with_validation(
    full_finetune, target_adapt_x, target_adapt_y, target_val_x, source_val_y,
    epochs=70, lr=7e-4, weight_decay=2e-4, seed=1221,
)
record_method(
    "Full fine-tune", full_finetune,
    task_parameters=count_parameters(full_finetune), note="all weights",
)

assert adaptation_results["Frozen features"]["trainable_parameters"] == 650
assert adaptation_results["Full fine-tune"]["trainable_parameters"] > 17000
print({name: {k: round(v, 3) if isinstance(v, float) else v for k, v in result.items()}
       for name, result in adaptation_results.items()})
```

</details>

目标准确率回答适配后的行为是否有效。适配模块处于启用状态时的干净源域准确率，回答部署配置是否破坏了旧行为。这与参数是否保留并不相同：即使 frozen base 在字节层面完全不变，启用的 head 或 adapter 也可能改变预测；反过来，完整 fine-tuning 也可能仍有很高的源域准确率，尽管每个存储权重都已经变成任务特定版本。

当目标集很小时，验证曲线通常比训练 loss 更有信息。目标训练 loss 接近零，仍可能伴随很差的目标验证准确率，因为大模型可以记住 80 个样本。更低学习率、更强 weight decay、更少解冻层或 PEFT 方法，可能比“训练更久”提供更合适的归纳偏置。

### **层冻结与判别式学习率** {#layer-freezing-discriminative-learning-rates}

部分 fine-tuning 在网络内部设置一条边界。下层保持稳定，上层与 head 负责适配。当目标域复用基础特征，但需要重新组合这些特征时，这种方式很有用。不过，冻结边界只是对“域特定信息位于哪里”的假设，不是通用规律；在严重的传感器或模态偏移下，早期层也可能必须适配。

判别式学习率（discriminative learning rates）进一步细化这一思想。如果参数组 $G_1,\ldots,G_K$ 分别使用学习率 $\eta_1,\ldots,\eta_K$，则一次更新为

$$
\theta_k \leftarrow \theta_k-\eta_k\nabla_{\theta_k}\mathcal{L}_T.
$$

为较低的预训练层设置更小的 $\eta_k$，相当于把它们视为更强的先验；为新初始化或任务特定层设置更大学习率，则允许它们快速移动。但这种 schedule 并不能自动保证安全，gradient scale、optimizer moment、normalization state 和训练时长也会共同决定实际位移 $\|\theta_k-\theta_k^{(0)}\|$。

<details>
<summary><strong>PyTorch：解冻上层 block 并使用判别式学习率</strong></summary>

```python
partial_tune = DigitClassifier()
partial_tune.load_state_dict(source_state)
for parameter in partial_tune.backbone.fc1.parameters():
    parameter.requires_grad = False

parameter_groups = [
    {"params": list(partial_tune.backbone.fc2.parameters()), "lr": 2e-4},
    {"params": list(partial_tune.head.parameters()), "lr": 1.5e-3},
]
train_with_validation(
    partial_tune, target_adapt_x, target_adapt_y, target_val_x, source_val_y,
    epochs=75, lr=1e-3, weight_decay=2e-4,
    parameter_groups=parameter_groups, seed=1222,
)
record_method("Partial + discriminative LR", partial_tune, note="fc2 + head")

frozen_gradients = [p.grad for p in partial_tune.backbone.fc1.parameters()]
assert all(gradient is None for gradient in frozen_gradients)
assert [group["lr"] for group in parameter_groups] == [2e-4, 1.5e-3]
print({
    "trainable parameters": count_parameters(partial_tune, True),
    "target test": round(adaptation_results["Partial + discriminative LR"]["target_accuracy"], 3),
    "clean source with adapted layers": round(adaptation_results["Partial + discriminative LR"]["source_accuracy_active"], 3),
})
```

</details>

有用的诊断包括每个参数组的 gradient norm、相对 checkpoint 的参数位移，以及逐步解冻 block 时的性能变化。如果新解冻 block 几乎没有梯度，额外容量并未真正使用；如果其位移迅速增大而验证结果下降，则学习率或目标证据缺少约束。Batch normalization 的 running statistics 需要特别处理：`requires_grad=False` 只会冻结 affine parameter，除非模块保持 evaluation mode，否则不会阻止 running mean 更新。

### **域适配与分布偏移** {#domain-adaptation-distribution-shift}

迁移学习常按任务组织，而**域适配（domain adaptation）**更关注分布。设源分布为 $P_S(x,y)$，目标分布为 $P_T(x,y)$：

- **Covariate shift：**$P_S(x)\neq P_T(x)$，但假定 $P(y\mid x)$ 稳定。
- **Label shift：**$P_S(y)\neq P_T(y)$，同时把类别条件输入分布视为稳定。
- **Concept shift：**$P_S(y\mid x)\neq P_T(y\mid x)$，证据的含义发生变化。

这些假设对应不同修正方法。Importance weighting 可以处理可识别的 covariate shift 或 label shift；表示对齐可以减少域特定变化；concept shift 通常需要新标签、修订目标或持续监控。如果源域与目标域类别比例不同，或者对齐导致类别混合，盲目匹配边缘特征反而可能有害。

![表示对齐前后的源域与目标域特征分布。](assets/dl12-domain-adaptation.svg){fig-align="center" width="74%" fig-alt="蓝色源域圆点与红色目标域方块从彼此分离的 cluster 移动到特征统计更重合的状态。"}

*根据 [Deep CORAL](https://mlanthology.org/eccv/2016/sun2016eccv-deep/) 提出的特征统计对齐目标绘制的原创教学图。*

CORAL 对齐二阶特征统计。对于中心化后的源特征 $H_S\in\mathbb{R}^{n_S\times d}$ 和目标特征 $H_T\in\mathbb{R}^{n_T\times d}$，定义 covariance matrix $C_S$ 与 $C_T$。Deep CORAL 加入

$$
\mathcal{L}_{\text{CORAL}}=\frac{1}{4d^2}\|C_S-C_T\|_F^2,
\qquad
\mathcal{L}=\mathcal{L}_{\text{source-cls}}+\lambda\mathcal{L}_{\text{CORAL}}.
$$

分类器仍由有标签源样本锚定，而无标签目标样本鼓励形成域不变特征。Frobenius norm 比较所有 covariance 元素，$4d^2$ 用于归一化尺度。这里演示的是机制，并不意味着 covariance matching 在所有问题上都最好。

<details>
<summary><strong>PyTorch：使用 Deep CORAL 进行无监督目标域对齐</strong></summary>

```python
def covariance(features):
    centered = features - features.mean(dim=0, keepdim=True)
    return centered.T @ centered / max(len(features) - 1, 1)


def coral_loss(source_features, target_features):
    dimension = source_features.shape[1]
    return (covariance(source_features) - covariance(target_features)).pow(2).sum() / (4 * dimension ** 2)


coral_model = DigitClassifier()
coral_model.load_state_dict(source_state)
optimizer = torch.optim.AdamW(coral_model.parameters(), lr=5e-4, weight_decay=2e-4)
source_alignment_x = source_train_x[:400]
source_alignment_y = source_train_y[:400]
best_state, best_val = copy.deepcopy(coral_model.state_dict()), -1.0
for _ in range(55):
    coral_model.train()
    optimizer.zero_grad()
    source_logits, source_features = coral_model(source_alignment_x, return_features=True)
    _, target_features = coral_model(target_adapt_x, return_features=True)
    classification = F.cross_entropy(source_logits, source_alignment_y)
    alignment = coral_loss(source_features, target_features)
    loss = classification + 12.0 * alignment
    loss.backward()
    optimizer.step()
    validation = accuracy(coral_model, target_val_x, source_val_y)
    if validation > best_val:
        best_val, best_state = validation, copy.deepcopy(coral_model.state_dict())
coral_model.load_state_dict(best_state)
record_method("CORAL (0 target labels)", coral_model, note="unlabeled target alignment")

with torch.no_grad():
    _, clean_features = source_model(source_alignment_x, return_features=True)
    _, shifted_features = source_model(target_adapt_x, return_features=True)
    initial_gap = float((covariance(clean_features) - covariance(shifted_features)).norm())
    _, aligned_source = coral_model(source_alignment_x, return_features=True)
    _, aligned_target = coral_model(target_adapt_x, return_features=True)
    final_gap = float((covariance(aligned_source) - covariance(aligned_target)).norm())
assert math.isfinite(final_gap)
print({
    "covariance gap before": round(initial_gap, 3),
    "covariance gap after": round(final_gap, 3),
    "target test": round(adaptation_results["CORAL (0 target labels)"]["target_accuracy"], 3),
})
```

</details>

监控对齐时，应观察各类别性能、calibration 和 subgroup behavior，而不只是 discrepancy 是否变小。坍缩表示可以匹配分布，却同时丢失任务信息。在生产中，drift detection 还必须判断偏移是否属于适配假设：新的标签政策不是传感器偏移。

### **多任务学习与持续学习** {#multi-task-continual-learning}

多任务学习（multi-task learning）联合优化相关目标。共享 backbone 从各任务特定 head 接收梯度：

$$
\mathcal{L}_{\text{MTL}}=\sum_{t=1}^{T}\alpha_t\mathcal{L}_t,
$$

其中 $\alpha_t$ 控制各任务的影响。共享特征可以构成 data-dependent regularization：数字身份与奇偶性都需要笔画证据。但任务也可能互相竞争。如果梯度 $g_i$ 与 $g_j$ 的 cosine similarity 为负，即 $g_i^\top g_j/(\|g_i\|\|g_j\|)<0$，那么一个任务的更新会在局部增大另一个任务的 loss。此时可能需要任务加权、gradient surgery、独立 normalization 或部分共享模块。

持续学习（continual learning）改变的是时间结构。任务或域依次到达，而旧数据可能不再可用。模型必须在适应新分布的**可塑性（plasticity）**与保留旧能力的**稳定性（stability）**之间平衡。三类主要方法是：

- **Replay：**保留或生成具有代表性的旧样本。
- **Regularization：**惩罚对旧任务重要的参数或输出发生变化。
- **Isolation：**分配任务特定 module、mask、adapter 或 expert。

下面的代码为相同低样本目标图像增加辅助 parity head。这不是额外数据，因为奇偶标签由数字身份确定性派生。示例展示一个 backbone 如何接收两种学习信号，以及如何检查它们的梯度兼容性。

<details>
<summary><strong>PyTorch：共享 backbone 的数字与奇偶性学习</strong></summary>

```python
class DigitParityModel(nn.Module):
    def __init__(self, pretrained_state):
        super().__init__()
        pretrained = DigitClassifier()
        pretrained.load_state_dict(pretrained_state)
        self.backbone = copy.deepcopy(pretrained.backbone)
        self.digit_head = copy.deepcopy(pretrained.head)
        self.parity_head = nn.Linear(64, 2)

    def forward(self, images):
        features = self.backbone(images)
        return self.digit_head(features), self.parity_head(features)


multitask_model = DigitParityModel(source_state)
optimizer = torch.optim.AdamW(multitask_model.parameters(), lr=7e-4, weight_decay=2e-4)
best_state, best_digit_val = copy.deepcopy(multitask_model.state_dict()), -1.0
for _ in range(70):
    multitask_model.train()
    optimizer.zero_grad()
    digit_logits, parity_logits = multitask_model(target_adapt_x)
    digit_loss = F.cross_entropy(digit_logits, target_adapt_y)
    parity_loss = F.cross_entropy(parity_logits, target_adapt_y % 2)
    (digit_loss + 0.35 * parity_loss).backward()
    optimizer.step()
    multitask_model.eval()
    with torch.no_grad():
        val_digit, _ = multitask_model(target_val_x)
        val_score = float((val_digit.argmax(1) == source_val_y).float().mean())
    if val_score > best_digit_val:
        best_digit_val, best_state = val_score, copy.deepcopy(multitask_model.state_dict())
multitask_model.load_state_dict(best_state)
multitask_model.eval()
with torch.no_grad():
    test_digit, test_parity = multitask_model(target_test_x)
    digit_score = float((test_digit.argmax(1) == source_test_y).float().mean())
    parity_score = float((test_parity.argmax(1) == source_test_y % 2).float().mean())

# Measure task-gradient cosine at the selected checkpoint.
digit_logits, parity_logits = multitask_model(target_adapt_x)
digit_grad = torch.autograd.grad(F.cross_entropy(digit_logits, target_adapt_y),
                                 multitask_model.backbone.parameters(), retain_graph=True)
parity_grad = torch.autograd.grad(F.cross_entropy(parity_logits, target_adapt_y % 2),
                                  multitask_model.backbone.parameters())
digit_vector = torch.cat([g.flatten() for g in digit_grad])
parity_vector = torch.cat([g.flatten() for g in parity_grad])
gradient_cosine = float(F.cosine_similarity(digit_vector, parity_vector, dim=0))
assert -1.0001 <= gradient_cosine <= 1.0001
print({"digit accuracy": round(digit_score, 3), "parity accuracy": round(parity_score, 3),
       "shared-gradient cosine": round(gradient_cosine, 3)})
```

</details>

只有当辅助目标改善了真正关心的性能边界时，它才有用。除非奇偶性本身就是部署需求，否则更高 parity score 不能补偿更差的数字识别。应该分别报告所有任务、检查任务梯度尺度，并测试推理时移除辅助 head 后收益是否仍然存在。

### **灾难性遗忘** {#catastrophic-forgetting}

灾难性遗忘（catastrophic forgetting）是学习新数据后，先前能力显著下降的现象。Gradient descent 只优化当前目标，并不会自动承担保留旧函数的义务。即使参数变化很小，也可能跨出狭窄 basin，或改变许多旧样本共同依赖的特征。

[Elastic Weight Consolidation](https://pubmed.ncbi.nlm.nih.gov/28292907/) 展示了一种 regularization 方法。它加入按参数重要性估计 $F_i$ 加权的二次惩罚：

$$
\mathcal{L}(\theta)=\mathcal{L}_{\text{new}}(\theta)
+\frac{\lambda}{2}\sum_i F_i(\theta_i-\theta_i^{*})^2.
$$

$\theta^{*}$ 是旧 checkpoint，较大的 $F_i$ 会阻止对旧任务重要的参数发生大幅移动。Replay 则通过把旧样本与新样本交替训练，以更直接的方式近似联合目标，但会引入内存、隐私与采样问题。

![朴素 fine-tuning 与 replay 约束适配的概念性源域能力保持轨迹。](assets/dl12-stability-plasticity.svg){fig-align="center" width="72%" fig-alt="目标域更新过程中，朴素 fine-tuning 的干净源域准确率迅速下降，而 replay 或约束使其保持在更高水平。"}

*原创概念图。实际能力保持程度必须针对具体任务和更新流测量；图中曲线不是实验结果。*

<details>
<summary><strong>PyTorch：使用 rehearsal 缓解源域遗忘</strong></summary>

```python
replay_local_idx = balanced_low_shot_indices(source_train_y, per_class=8, seed=1225)
replay_x = source_train_x[replay_local_idx]
replay_y = source_train_y[replay_local_idx]

replay_model = DigitClassifier()
replay_model.load_state_dict(source_state)
optimizer = torch.optim.AdamW(replay_model.parameters(), lr=7e-4, weight_decay=2e-4)
best_state, best_joint = copy.deepcopy(replay_model.state_dict()), -1.0
for _ in range(70):
    replay_model.train()
    optimizer.zero_grad()
    target_loss = F.cross_entropy(replay_model(target_adapt_x), target_adapt_y)
    old_loss = F.cross_entropy(replay_model(replay_x), replay_y)
    (target_loss + 0.6 * old_loss).backward()
    optimizer.step()
    target_val = accuracy(replay_model, target_val_x, source_val_y)
    source_val = accuracy(replay_model, source_val_x, source_val_y)
    joint_score = target_val + 0.35 * source_val
    if joint_score > best_joint:
        best_joint, best_state = joint_score, copy.deepcopy(replay_model.state_dict())
replay_model.load_state_dict(best_state)
record_method("Full tune + replay", replay_model, note="80 stored source examples")

naive_source = adaptation_results["Full fine-tune"]["source_accuracy_active"]
replay_source = adaptation_results["Full tune + replay"]["source_accuracy_active"]
print({
    "naive target/source": (
        round(adaptation_results["Full fine-tune"]["target_accuracy"], 3), round(naive_source, 3)
    ),
    "replay target/source": (
        round(adaptation_results["Full tune + replay"]["target_accuracy"], 3), round(replay_source, 3)
    ),
})
```

</details>

遗忘应该通过矩阵而不是一个最终数字来测量：每次更新后都评估此前的每项任务。还要区分遗忘与普通分布不匹配。如果干净 checkpoint 在任何更新前就无法处理目标域，这个差距不是遗忘；如果目标适配后干净域性能下降，才是 retention loss。Adapter isolation 可以精确保留 base 参数，但路由到错误 adapter 仍然会导致行为失败。

### **为什么参数效率很重要** {#why-parameter-efficiency-matters}

完整 fine-tuning 会为每项任务存储并优化所有参数的特定副本。对于含有 $N$ 个参数的模型，mixed-precision Adam 类训练可能需要低精度权重、梯度、FP32 master weight 和两个 FP32 moment buffer。精确口径依赖具体实现，但 optimizer state 可能比 checkpoint 本身更大。如果存在 $K$ 个客户或域，保存 $K$ 份完整模型将按 $O(KN)$ 扩展。

参数高效微调（parameter-efficient fine-tuning，PEFT）冻结共享 base，只学习 $m\ll N$ 个任务参数。它针对三种不同成本，这些成本不能混为一谈：

- **可训练参数：**决定 gradient 和 optimizer-state storage。
- **每任务存储参数：**决定多个变体共同存在时的成本。
- **运行时内存与延迟：**仍然包含 frozen base 和 activation；可训练参数更少并不意味着 base 没有成本。

![Adapter、LoRA 和 prompt 方法在不同位置注入任务特定容量。](assets/dl12-peft-methods.svg){fig-align="center" width="76%" fig-alt="三个 panel 分别展示 frozen block 后的 adapter bottleneck、frozen weight 旁的低秩矩阵，以及 frozen model 前的可学习 prompt vector。"}

*根据 [Houlsby adapters](https://proceedings.mlr.press/v97/houlsby19a.html)、[LoRA](https://arxiv.org/abs/2106.09685)、[prompt tuning](https://aclanthology.org/2021.emnlp-main.243/) 与 [prefix tuning](https://aclanthology.org/2021.acl-long.353/) 综合绘制的原创教学图。*

<details>
<summary><strong>Python：核算参数量与近似 AdamW state</strong></summary>

```python
total_parameters = count_parameters(source_model)
head_parameters = sum(p.numel() for p in source_model.head.parameters())


def approximate_training_bytes(trainable_parameters, parameter_bytes=4,
                               gradient_bytes=4, adam_moment_bytes=8):
    # This excludes activations, allocator overhead, temporary kernels, and the frozen base.
    return trainable_parameters * (parameter_bytes + gradient_bytes + adam_moment_bytes)


resource_rows = {
    "full fine-tune": (total_parameters, total_parameters),
    "head only": (head_parameters, head_parameters),
    "partial fc2 + head": (count_parameters(partial_tune, True), count_parameters(partial_tune, True)),
}
for name, (trainable, per_task) in resource_rows.items():
    print({
        "method": name,
        "trainable fraction": round(trainable / total_parameters, 4),
        "approx train-state KiB": round(approximate_training_bytes(trainable) / 1024, 1),
        "per-task FP32 KiB": round(per_task * 4 / 1024, 1),
    })

assert head_parameters == 650
assert approximate_training_bytes(total_parameters) > approximate_training_bytes(head_parameters)
```

</details>

当大型 base 要在许多任务间复用、optimizer memory 是瓶颈，或任务 module 必须独立切换时，PEFT 最有价值。当模型本身已经很小、目标域完全不同，或部署仍要求把每个变体合并成独立 binary 时，其优势会减弱。参数量只是约束，不是最终目标；目标质量、retention、calibration 和运维简洁性仍然更重要。

### **基于 Adapter 的微调** {#adapter-based-fine-tuning}

Adapter 在 frozen network 中插入一个小型 residual bottleneck。对于 hidden state $h\in\mathbb{R}^{d}$ 和 bottleneck width $b\ll d$，

$$
\operatorname{Adapter}(h)=h+W_{\text{up}}\,\sigma(W_{\text{down}}h),
$$

其中 $W_{\text{down}}\in\mathbb{R}^{b\times d}$，$W_{\text{up}}\in\mathbb{R}^{d\times b}$。把 up projection 初始化为接近零，会使模块开始时接近 identity residual，从而避免适配过程立即覆盖预训练函数。[Houlsby 等人](https://proceedings.mlr.press/v97/houlsby19a.html)为 Transformer transfer 展示了这种设计，但该机制适用于任何 hidden representation。

Adapter 提供显式任务模块化：保留一个不可变 base，为每个域加载小型 module。它的成本包括额外 sequential operation、adapter placement 选择，以及需要批处理多个 adapter 时的 serving complexity。Bottleneck width 在容量、存储与延迟之间形成权衡。

<details>
<summary><strong>PyTorch：插入并训练 residual bottleneck adapter</strong></summary>

```python
class AdapterClassifier(nn.Module):
    def __init__(self, pretrained_state, bottleneck=8):
        super().__init__()
        pretrained = DigitClassifier()
        pretrained.load_state_dict(pretrained_state)
        self.backbone = copy.deepcopy(pretrained.backbone)
        self.head = copy.deepcopy(pretrained.head)
        for parameter in self.backbone.parameters():
            parameter.requires_grad = False
        self.adapter = nn.Sequential(
            nn.Linear(64, bottleneck), nn.ReLU(), nn.Linear(bottleneck, 64)
        )
        nn.init.zeros_(self.adapter[-1].weight)
        nn.init.zeros_(self.adapter[-1].bias)

    def forward(self, images):
        features = self.backbone(images)
        adapted = features + self.adapter(features)
        return self.head(adapted)


adapter_model = AdapterClassifier(source_state, bottleneck=8)
train_with_validation(
    adapter_model, target_adapt_x, target_adapt_y, target_val_x, source_val_y,
    epochs=75, lr=2e-3, weight_decay=2e-4, seed=1230,
)
adapter_task_parameters = sum(p.numel() for p in adapter_model.adapter.parameters()) + sum(
    p.numel() for p in adapter_model.head.parameters()
)
record_method("Adapter", adapter_model, task_parameters=adapter_task_parameters, note="bottleneck=8 + head")

assert all(not p.requires_grad for p in adapter_model.backbone.parameters())
assert adapter_task_parameters < total_parameters
print({
    "adapter + head parameters": adapter_task_parameters,
    "fraction of full model": round(adapter_task_parameters / total_parameters, 4),
    "target test": round(adaptation_results["Adapter"]["target_accuracy"], 3),
})
```

</details>

调试应从 identity behavior 开始：训练前，zero-initialized adapter 的输出应该在数值精度内重现 base output。训练时要确认 base gradient 保持 `None`、adapter norm 离开零，并验证性能提升不是 evaluation mode 改变造成的。如果性能饱和，可以先加宽 bottleneck 或在更多层放置 adapter，再判断是否真的需要完整 fine-tuning。

### **LoRA** {#lora}

Low-Rank Adaptation（LoRA）无需存储 dense matrix，就能表示任务特定权重更新。对于 frozen $W_0\in\mathbb{R}^{d_{out}\times d_{in}}$，

$$
y=W_0x+\Delta Wx,
\qquad
\Delta W=\frac{\alpha}{r}BA,
$$

其中 $A\in\mathbb{R}^{r\times d_{in}}$，$B\in\mathbb{R}^{d_{out}\times r}$，rank $r\ll\min(d_{in},d_{out})$。可训练参数量从 $d_{out}d_{in}$ 变为 $r(d_{in}+d_{out})$。缩放项 $\alpha/r$ 把更新幅度与 rank 分离。把 $B$ 初始化为零会让 $\Delta W=0$，同时随机 $A$ 允许梯度先进入 $B$。

LoRA 的假设不是预训练权重本身是低秩的，而是**任务特定的参数位移**通常可以在低维子空间中表示。[Hu 等人](https://arxiv.org/abs/2106.09685)最初面向大型语言模型提出 LoRA，并强调学习到的更新可以合并进 $W_0$ 进行推理，从而避免 adapter 带来的额外 sequential layer。

<details>
<summary><strong>PyTorch：为数字 backbone 实现 LoRA</strong></summary>

```python
class LoRALinear(nn.Module):
    def __init__(self, base_layer, rank=4, alpha=8.0):
        super().__init__()
        self.base = copy.deepcopy(base_layer)
        for parameter in self.base.parameters():
            parameter.requires_grad = False
        self.rank = rank
        self.scale = alpha / rank
        self.A = nn.Parameter(torch.empty(rank, base_layer.in_features))
        self.B = nn.Parameter(torch.zeros(base_layer.out_features, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    def forward(self, inputs):
        low_rank = F.linear(F.linear(inputs, self.A), self.B)
        return self.base(inputs) + self.scale * low_rank

    def merged_weight(self):
        return self.base.weight + self.scale * (self.B @ self.A)


class LoRAClassifier(nn.Module):
    def __init__(self, pretrained_state, rank=4):
        super().__init__()
        pretrained = DigitClassifier()
        pretrained.load_state_dict(pretrained_state)
        self.fc1 = LoRALinear(pretrained.backbone.fc1, rank=rank)
        self.fc2 = LoRALinear(pretrained.backbone.fc2, rank=rank)
        self.head = copy.deepcopy(pretrained.head)

    def forward(self, images):
        flat = images.flatten(1)
        features = F.relu(self.fc2(F.relu(self.fc1(flat))))
        return self.head(features)


lora_model = LoRAClassifier(source_state, rank=4)
with torch.no_grad():
    base_difference = float((lora_model(target_adapt_x) - source_model(target_adapt_x)).abs().max())
train_with_validation(
    lora_model, target_adapt_x, target_adapt_y, target_val_x, source_val_y,
    epochs=75, lr=1.8e-3, weight_decay=2e-4, seed=1231,
)
lora_task_parameters = count_parameters(lora_model, True)
record_method("LoRA rank 4", lora_model, task_parameters=lora_task_parameters, note="two LoRA layers + head")

assert base_difference < 1e-6
assert lora_task_parameters < total_parameters
assert lora_model.fc1.merged_weight().shape == lora_model.fc1.base.weight.shape
print({
    "initial max output difference": base_difference,
    "trainable parameters": lora_task_parameters,
    "target test": round(adaptation_results["LoRA rank 4"]["target_accuracy"], 3),
})
```

</details>

Rank 是容量超参数，不是质量保证。Rank 太小会使所需位移欠拟合；太大则削弱存储优势并可能过拟合。Layer selection 可能比统一 rank 更重要。应该监控 $\|BA\|$、在固定参数预算下比较不同 rank，并在部署前数值验证 merge equivalence。合并后，如果不另外保留 base 或 delta，就会失去任务切换灵活性。

### **QLoRA** {#qlora}

QLoRA 在训练 LoRA 参数的同时，降低 **frozen base** 的内存占用。在 [Dettmers 等人](https://proceedings.neurips.cc/paper_files/paper/2023/hash/1feb87871436031bdc0f2beaa62a049b-Abstract-Conference.html)提出的完整方法中，base weight 使用 4-bit NormalFloat（NF4），quantization constant 通过 double quantization 再次压缩，paged optimizer 用于处理内存峰值。计算时把 base block 反量化到适当的 compute dtype；梯度通过这些操作流入 LoRA 参数，而 quantized base 保持冻结。

![LoRA 保留浮点 frozen base，而 QLoRA 用 4-bit 形式存储 base，并学习相同的低秩路径。](assets/dl12-lora-qlora.svg){fig-align="center" width="76%" fig-alt="两个 panel 对比浮点 frozen base 加 LoRA matrix，以及计算时反量化的 four-bit frozen base 加 LoRA matrix。"}

*根据 [LoRA 论文](https://arxiv.org/abs/2106.09685)和 [QLoRA NeurIPS 论文](https://proceedings.neurips.cc/paper_files/paper/2023/hash/1feb87871436031bdc0f2beaa62a049b-Abstract-Conference.html)绘制的原创教学图。*

下面的可运行示例刻意缩小范围：它使用**按输出行对称 int4 quantization** 来展示数据路径。它不是 NF4，没有实现 double quantization 或 paged optimizer，因此不应被称为生产级 QLoRA 实现。这个区分很重要，因为“4-bit weight 加 LoRA”遗漏了构成 QLoRA 内存与质量表现的多项关键技术。

<details>
<summary><strong>PyTorch：演示 quantized frozen base 与 LoRA 路径</strong></summary>

```python
def symmetric_int4_quantize(weight):
    scale = weight.abs().amax(dim=1, keepdim=True).clamp_min(1e-8) / 7.0
    quantized = torch.round(weight / scale).clamp(-7, 7).to(torch.int8)
    return quantized, scale


class QuantizedLoRALinear(nn.Module):
    def __init__(self, base_layer, rank=4, alpha=8.0):
        super().__init__()
        quantized, scale = symmetric_int4_quantize(base_layer.weight.detach())
        self.register_buffer("qweight", quantized)
        self.register_buffer("weight_scale", scale)
        self.register_buffer("base_bias", base_layer.bias.detach().clone())
        self.rank = rank
        self.lora_scale = alpha / rank
        self.A = nn.Parameter(torch.empty(rank, base_layer.in_features))
        self.B = nn.Parameter(torch.zeros(base_layer.out_features, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    def forward(self, inputs):
        dequantized = self.qweight.float() * self.weight_scale
        base_output = F.linear(inputs, dequantized, self.base_bias)
        return base_output + self.lora_scale * F.linear(F.linear(inputs, self.A), self.B)


class ToyQLoRAClassifier(nn.Module):
    def __init__(self, pretrained_state, rank=4):
        super().__init__()
        pretrained = DigitClassifier()
        pretrained.load_state_dict(pretrained_state)
        self.fc1 = QuantizedLoRALinear(pretrained.backbone.fc1, rank=rank)
        self.fc2 = QuantizedLoRALinear(pretrained.backbone.fc2, rank=rank)
        self.head = copy.deepcopy(pretrained.head)

    def forward(self, images):
        flat = images.flatten(1)
        features = F.relu(self.fc2(F.relu(self.fc1(flat))))
        return self.head(features)


qlora_demo = ToyQLoRAClassifier(source_state, rank=4)
dequantized_fc1 = qlora_demo.fc1.qweight.float() * qlora_demo.fc1.weight_scale
reference_fc1 = source_model.backbone.fc1.weight.detach()
relative_error = float((dequantized_fc1 - reference_fc1).norm() / reference_fc1.norm())
train_with_validation(
    qlora_demo, target_adapt_x, target_adapt_y, target_val_x, source_val_y,
    epochs=75, lr=1.8e-3, weight_decay=2e-4, seed=1232,
)
qlora_task_parameters = count_parameters(qlora_demo, True)
record_method("Toy int4 + LoRA", qlora_demo, task_parameters=qlora_task_parameters,
              note="symmetric int4 demo, not NF4 QLoRA")

assert qlora_demo.fc1.qweight.dtype == torch.int8
assert qlora_demo.fc1.qweight.requires_grad is False
assert qlora_task_parameters == lora_task_parameters
print({
    "fc1 relative quantization error": round(relative_error, 4),
    "trainable parameters": qlora_task_parameters,
    "target test": round(adaptation_results["Toy int4 + LoRA"]["target_accuracy"], 3),
})
```

</details>

Quantization 引入第二个近似维度。适配前就应该诊断 frozen quantized checkpoint、比较量化与未量化 logits，并识别敏感层。QLoRA 会减少 base-weight storage 和 optimizer 压力，但 activation、LoRA state、dequantization buffer、sequence length 和 kernel support 仍然决定训练能否装入内存。

### **Prompt Tuning 与 Prefix Tuning** {#prompt-tuning-prefix-tuning}

Prompt 方法通过学习连续输入而不是 weight delta 来适配 frozen model。**Prompt tuning** 在输入处前置或注入可训练 embedding $P\in\mathbb{R}^{m\times d}$。对 token sequence $X$，模型接收 $[P;X]$。[Lester 等人](https://aclanthology.org/2021.emnlp-main.243/)发现，随着语言模型规模增大，prompt tuning 与完整 fine-tuning 的差距会缩小。

**Prefix tuning** 在多个 Transformer layer 中注入学习到的 key/value-like state。如果某层原本 attention 到 $K,V$，则改为使用

$$
K'=[P_K;K],\qquad V'=[P_V;V].
$$

Prefix 可以在不修改 base projection 的情况下影响每一层 attention。它比 input-only prompt 更深入，同时会增加 attention length，从而提高 key/value memory 与计算量。[Li 和 Liang](https://aclanthology.org/2021.acl-long.353/)面向 conditional generation 提出了这一方法。

数字 MLP 没有 token sequence 或 attention cache，因此合适的类比是 **visual prompt**：学习一个 8×8 additive pattern，应用于每个目标图像，同时保持分类器冻结。这个示例演示 prompt-only adaptation，而不是 Transformer prefix tuning。[Visual Prompt Tuning](https://www.ecva.net/papers/eccv_2022/papers_ECCV/html/4175_ECCV_2022_paper.php)提供了经典视觉场景背景。

<details>
<summary><strong>PyTorch：冻结分类器并学习 visual prompt</strong></summary>

```python
class VisualPromptClassifier(nn.Module):
    def __init__(self, pretrained_state):
        super().__init__()
        self.base = DigitClassifier()
        self.base.load_state_dict(pretrained_state)
        for parameter in self.base.parameters():
            parameter.requires_grad = False
        self.prompt = nn.Parameter(torch.zeros(1, 1, 8, 8))

    def forward(self, images):
        prompted = (images + 0.35 * torch.tanh(self.prompt)).clamp(0.0, 1.0)
        return self.base(prompted)


prompt_model = VisualPromptClassifier(source_state)
train_with_validation(
    prompt_model, target_adapt_x, target_adapt_y, target_val_x, source_val_y,
    epochs=100, lr=2e-2, weight_decay=1e-4, seed=1233,
)
prompt_parameters = prompt_model.prompt.numel()
record_method("Visual prompt", prompt_model, task_parameters=prompt_parameters,
              note="64-pixel additive prompt")

assert prompt_parameters == 64
assert all(not p.requires_grad for p in prompt_model.base.parameters())
assert float(prompt_model.prompt.detach().abs().max()) > 0
print({
    "prompt parameters": prompt_parameters,
    "target test": round(adaptation_results["Visual prompt"]["target_accuracy"], 3),
    "clean source with prompt active": round(adaptation_results["Visual prompt"]["source_accuracy_active"], 3),
})
```

</details>

Prompt 方法可以极其紧凑，但其效果强烈依赖模型规模、prompt length、初始化，以及预训练是否让模型可以通过输入接口被控制。它们还可能占用 context position 或增加 key/value cache。只有配合特定 tokenizer、template 或图像 normalization 才有效的 prompt，在运维上也与该预处理契约紧密耦合。

### **完整微调与参数高效适配的比较** {#full-fine-tuning-parameter-efficient-adaptation}

不存在普遍占优的适配方法。完整 fine-tuning 提供最大直接容量，却会复制模型并可能遗忘；frozen feature 稳定且成本低，却不能修复表示；adapter 添加模块化 residual computation；LoRA 提供可合并的低秩 weight update；QLoRA 进一步压缩训练期间的 frozen base；prompt 方法把任务容量放进 learned input，但可能消耗 context 或依赖模型规模。

下表由共享实验生成。270 个测试图像上的准确率差异会受噪声与随机种子影响，只用于说明机制，不是 leaderboard。`source_accuracy_active` 在任务特定 module 保持启用时评估干净源域。`task_parameters` 统计该教学实现中每项任务必须存储的参数，不包括只保存一次的共享 base。

<details>
<summary><strong>Python：综合质量、能力保持与存储证据</strong></summary>

```python
ordered_methods = [
    "Frozen features", "Partial + discriminative LR", "Full fine-tune",
    "Full tune + replay", "CORAL (0 target labels)", "Adapter",
    "LoRA rank 4", "Toy int4 + LoRA", "Visual prompt",
]
comparison_rows = []
for method in ordered_methods:
    result = adaptation_results[method]
    comparison_rows.append({
        "method": method,
        "target_acc": round(result["target_accuracy"], 3),
        "clean_acc_active": round(result["source_accuracy_active"], 3),
        "trainable": result["trainable_parameters"],
        "per_task_KiB_FP32": round(result["task_parameters"] * 4 / 1024, 2),
        "note": result["note"],
    })

header = f'{"method":<30} {"target":>7} {"clean":>7} {"trainable":>10} {"task KiB":>9}'
print(header)
print("-" * len(header))
for row in comparison_rows:
    print(f'{row["method"]:<30} {row["target_acc"]:>7.3f} {row["clean_acc_active"]:>7.3f} '
          f'{row["trainable"]:>10d} {row["per_task_KiB_FP32"]:>9.2f}')

assert len(comparison_rows) == 9
assert next(r for r in comparison_rows if r["method"] == "Visual prompt")["trainable"] == 64
assert next(r for r in comparison_rows if r["method"] == "Full fine-tune")["per_task_KiB_FP32"] > next(
    r for r in comparison_rows if r["method"] == "LoRA rank 4"
)["per_task_KiB_FP32"]
```

</details>

| 策略 | 表示可以移动吗？ | 每任务 artifact | 推理影响 | 典型风险 |
|---|---:|---|---|---|
| Frozen features | 否 | Head | 共享 backbone 加所选 head | 目标信息无法线性访问 |
| Partial tuning | 上层可以 | 更新层与 head | 通常是任务特定 checkpoint | 冻结边界错误 |
| Full fine-tuning | 可以，所有层 | 完整模型 | 没有额外 module | 过拟合、遗忘、存储成本 |
| Adapter | 通过 residual module | Adapter 与 head | 增加 sequential operation | Bottleneck 或 placement 欠拟合 |
| LoRA | 通过低秩 weight delta | $A,B$ 与 head | 可合并或可切换 | Rank 或 layer coverage 不足 |
| QLoRA | 使用相同 LoRA delta；base 被量化 | LoRA 与 quantization metadata | 取决于 merge 和 quantized kernel | 量化敏感性 |
| Prompt/prefix | Base 冻结；input/state 移动 | 学习到的 prompt vector | 增加 token、K/V 或 preprocessing | 可控性弱、context 成本 |

实际选择可以遵循以下顺序：

1. 建立 zero-shot 与 frozen-feature baseline。
2. 如果仍有表示不匹配，尝试 partial tuning 或适度的 PEFT module。
3. 在相同标签、验证预算和 checkpoint 规则下比较。
4. 测量目标质量、源域能力保持、calibration、训练内存、任务存储与 serving latency。
5. 只有完整 fine-tuning 的质量增益足以抵偿运维成本时，才升级到它。

方法选择本身就是部署架构选择。合并后的 LoRA 可能适合单个静态模型；可切换 adapter 可能适合多 tenant；当只能访问 embedding interface 时，prompt tuning 可能更方便；拥有大量目标证据的单个高价值域，则可能值得完整 tuning。

### **章节比较与总结** {#chapter-comparison-summary}

预训练提供可复用起点；迁移是“这个起点能降低目标风险”的经验性主张；适配则控制函数发生多少变化。本章的递进关系形成一条容量阶梯：

- **Frozen features** 检验目标信息是否已经暴露出来。
- **Partial tuning 与 full tuning** 逐步放松源 checkpoint 先验。
- **Domain alignment** 利用分布结构，有时甚至不需要目标标签。
- **Replay 或 regularization** 在顺序更新中保护既有能力。
- **Adapter、LoRA、QLoRA 与 prompt 方法** 把任务特定容量限制在紧凑 artifact 中。

单纯比较参数量时，几个关键区别很容易丢失。Frozen parameter 仍占用推理内存；保留 base byte 并不保证启用任务 module 后的行为也被保留；QLoRA 不等于泛化的 int4 quantization 加 low-rank layer；prompt tuning 与 prefix tuning 在不同位置注入 learned state；domain adaptation 也只有在明确的 shift assumption 下才成立。

对一个新项目，至少保留四类证据：无适配 baseline、完整 tuning 的上限参考、至少一个参数高效替代方案，以及对重要源行为的 retention evaluation。记录标签在哪里进入、checkpoint 如何选择，以及每项任务必须存储什么。这样，adaptation 就不再只是流行方法名，而会成为关于**容量、证据、稳定性与系统成本**的可审计决策。

第 13 章会把这个问题扩展到更大规模。Foundation model 让一个 pretrained base 服务于多项任务和多种模态，因此 data curation、scaling behavior、model routing 与 lifecycle governance 会与本章研究的局部适配规则同等重要。